# 1 — Setup
One-time **sample-based dataset qualification** for *Image Enhancement as Preprocessing for Industrial Anomaly Detection Under Challenging Illumination*.
Run **Runtime → Run all** in a fresh Colab CPU runtime (Hardware accelerator: **None**) and authorize Drive access. Only the output directory is written.
The summary is marked incomplete before inspection so an interrupted run cannot leave a current PASS.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import json
import os
import platform
import random
import time
import PIL
from PIL import Image

# All configurable paths are here; do not point output inside a dataset.
dataRoot = Path('/content/drive/MyDrive/ColabData/CS7367_Vision')
datasetRoots = {'MVTec AD 2': dataRoot / 'mvtec_ad_2',
                'VisA': dataRoot / 'VisA_20220922'}
outputRoot = dataRoot / 'output' / 'validation'
sampleSeed = 7367
sampleLimit = 15  # Per category, across ALL image groups; separate mask budget.
for datasetRoot in datasetRoots.values():
    if outputRoot.resolve().is_relative_to(datasetRoot.resolve()):
        raise ValueError('Output must be outside both dataset roots.')
outputRoot.mkdir(parents=True, exist_ok=True)
runStartedUtc = datetime.now(timezone.utc).isoformat()
exceptions = []
inventory = []
imageRecords = []
maskRecords = []
maskPairs = []
annotationCounts = Counter()
filesChecked = {name: Counter(images=0, masks=0, pairedImageHeaders=0) for name in datasetRoots}
inventoryComplete = {name: False for name in datasetRoots}
integrityComplete = {name: False for name in datasetRoots}
accessibleRoots = {}
limitations = {
    'MVTec AD 2': ['Private test ground truth is intentionally non-public. Missing private image partitions are recorded as scope limitations; present private partitions must be nonempty and their sampled images readable.'],
    'VisA': ['Original release has Normal/Anomaly groups, not native train/test partitions. Normal-image masks are intentionally omitted. No split is created.']}
Image.init()
inventoryFields = ['dataset', 'category', 'partition', 'imageCount', 'annotationCount', 'maskCount',
                   'sampledImageCount', 'sampledMaskCount', 'imagePaths', 'maskPaths',
                   'sampledImagePaths', 'sampledMaskPaths']
sampledImageRecords = []
sampledMaskRecords = []
samplesComplete = False
qualificationBasis = 'sample-based qualification; not exhaustive image integrity certification'
residualRisk = 'Unsampled images or masks may be corrupt or inconsistent; this run cannot certify their integrity.'
for dataset in limitations:
    limitations[dataset].append(residualRisk)

def addIssue(dataset, category, partition, filePath, description, severity='critical'):
    exceptions.append(dict(dataset=dataset, category=category, partition=partition,
                           filePath=str(filePath), severity=severity, issueDescription=description))

def listImages(folder):
    foundFiles = []
    def raiseWalkError(error):
        raise error
    if folder.is_dir():
        for currentRoot, folderNames, fileNames in os.walk(folder, onerror=raiseWalkError):
            folderNames.sort()
            for fileName in sorted(fileNames):
                filePath = Path(currentRoot) / fileName
                if not fileName.startswith('.'):
                    # Retain all filenames in these directories for inventory and sampling.
                    foundFiles.append(filePath)
    return foundFiles

def findRoot(root, expectedCategories):
    # Permit an extraction wrapper, but never select between ambiguous copies.
    candidates = [root] if root.is_dir() and any((root / c).is_dir() for c in expectedCategories) else []
    if root.is_dir() and not candidates:
        candidates = sorted({p.parent for c in expectedCategories for p in root.rglob(c) if p.is_dir()})
    if len(candidates) != 1:
        raise ValueError(f'Expected one category root below {root}; found {len(candidates)}.')
    return candidates[0]

def addGroup(dataset, category, partition, folder, required=True):
    groupImages = listImages(folder)
    if required and not groupImages:
        addIssue(dataset, category, partition, folder, 'Required image directory is missing or empty.')
    imageRecords.extend((dataset, category, partition, p) for p in groupImages)
    inventory.append(dict(dataset=dataset, category=category, partition=partition,
                          imageCount=len(groupImages), annotationCount=0, maskCount=0))
    return groupImages

# Invalidate earlier evidence before any inventory work.
for csvName, fieldNames in [('dataset_inventory.csv', inventoryFields),
                            ('validation_exceptions.csv', ['dataset', 'category', 'partition', 'filePath', 'severity', 'issueDescription'])]:
    with (outputRoot / csvName).open('w', newline='', encoding='utf-8') as handle:
        csv.DictWriter(handle, fieldnames=fieldNames).writeheader()
with (outputRoot / 'validation_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'validationTimestampUtc': runStartedUtc, 'complete': False,
               'overallResult': 'FAIL', 'qualificationBasis': qualificationBasis, 'sampleSeed': sampleSeed, 'sampleLimitPerCategory': sampleLimit, 'reason': 'Validation started; completion evidence not yet written.'}, handle, indent=2)
for dataset, root in datasetRoots.items():
    try:
        if not root.is_dir():
            raise FileNotFoundError(root)
        next(root.iterdir(), None)
        print(f'{dataset}: accessible — {root}')
    except Exception as error:
        addIssue(dataset, '', '', root, f'Dataset root inaccessible: {error}')
        print(f'{dataset}: FAIL — root inaccessible')


# 2 — Dataset Inventory
[MVTec documentation](https://www.mvtec.com/research-teaching/datasets/mvtec-ad-2) and its linked official code utilities define `train/good`, `validation/good`, `test_public/{good,bad}`, and flat private image partitions. Public anomaly masks are `test_public/ground_truth/bad/<stem>_mask.png`.
Primary categories: **can, wallplugs, sheet_metal**. All eight categories are checked.
[VisA documentation](https://github.com/amazon-science/spot-diff) defines the original category-level `Data/Images/{Normal,Anomaly}`, `Data/Masks/Anomaly`, and `image_anno.csv` (also called `image_annot.csv` in its prose). All 12 official categories and any additional discovered categories are inventoried. Private MVTec image partitions may be absent in a public-only installation; this limits qualification scope and is explicitly reported.


In [ ]:
mvtecCategories = ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']
visaCategories = ['candle', 'capsules', 'cashew', 'chewinggum', 'fryum', 'macaroni1', 'macaroni2', 'pcb1', 'pcb2', 'pcb3', 'pcb4', 'pipe_fryum']
visaCategoryRoots = []
for dataset, expectedCategories in [('MVTec AD 2', mvtecCategories), ('VisA', visaCategories)]:
    inventoryStarted = time.monotonic()
    print(f'{dataset}: inventory START (all filenames; no decoding)', flush=True)
    try:
        root = findRoot(datasetRoots[dataset], expectedCategories)
        accessibleRoots[dataset] = root
        if dataset == 'MVTec AD 2':
            discoveredCategories = {p.name for p in root.iterdir() if p.is_dir() and any((p / s).is_dir() for s in ['train', 'validation', 'test_public', 'test_private', 'test_private_mixed'])}
        else:
            discoveredCategories = {p.name for p in root.iterdir() if p.is_dir() and (p / 'Data').is_dir()}
        for category in sorted(set(expectedCategories) | discoveredCategories):
            print(f'{dataset}/{category}: inventory START', flush=True)
            categoryRoot = root / category
            if not categoryRoot.is_dir():
                addIssue(dataset, category, '', categoryRoot, 'Missing required category.')
            if category not in expectedCategories:
                addIssue(dataset, category, '', categoryRoot, 'Unexpected category; requires review.')
            if dataset == 'MVTec AD 2':
                for partition in ['train', 'validation', 'test_public', 'test_private', 'test_private_mixed']:
                    partitionRoot = categoryRoot / partition
                    if 'private' in partition:
                        if not partitionRoot.exists():
                            addIssue(dataset, category, partition, partitionRoot, 'Private image partition unavailable; excluded from qualification scope.', 'limitation')
                            inventory.append(dict(dataset=dataset, category=category, partition=partition, imageCount=0, annotationCount=0, maskCount=0))
                            continue
                        groupImages = addGroup(dataset, category, partition, partitionRoot)
                        for imagePath in groupImages:
                            if imagePath.parent != partitionRoot:
                                addIssue(dataset, category, partition, imagePath, 'Private images must be directly inside the partition.')
                    else:
                        groupImages = []
                        for label in (['good', 'bad'] if partition == 'test_public' else ['good']):
                            groupImages += addGroup(dataset, category, partition + '/' + label, partitionRoot / label)
                        maskRoot = partitionRoot / 'ground_truth'
                        for maskPath in listImages(maskRoot):
                            maskRecords.append((dataset, category, partition + '/bad', maskPath))
                        for imagePath in groupImages:
                            if imagePath.parent.name == 'bad':
                                maskPath = maskRoot / 'bad' / (imagePath.stem + '_mask.png')
                                maskPairs.append((dataset, category, partition + '/bad', imagePath, maskPath))
                        expectedImages = set(groupImages)
                        for imagePath in listImages(partitionRoot):
                            if not imagePath.is_relative_to(maskRoot) and imagePath not in expectedImages:
                                imageRecords.append((dataset, category, partition, imagePath))
                                inventory.append(dict(dataset=dataset, category=category, partition=partition, imageCount=1, annotationCount=0, maskCount=0))
                                addIssue(dataset, category, partition, imagePath, 'Image outside the documented good/bad layout.')
            else:
                visaCategoryRoots.append(categoryRoot)
                for label in ['Normal', 'Anomaly']:
                    addGroup(dataset, category, label, categoryRoot / 'Data' / 'Images' / label)
                for maskPath in listImages(categoryRoot / 'Data' / 'Masks'):
                    maskRecords.append((dataset, category, 'Anomaly', maskPath))
                knownImages = {r[3] for r in imageRecords if r[0] == dataset and r[1] == category}
                for imagePath in listImages(categoryRoot / 'Data' / 'Images'):
                    if imagePath not in knownImages:
                        imageRecords.append((dataset, category, 'unexpected', imagePath))
                        inventory.append(dict(dataset=dataset, category=category, partition='unexpected', imageCount=1, annotationCount=0, maskCount=0))
                        addIssue(dataset, category, '', imagePath, 'Image outside Normal/Anomaly groups.')
            print(f'{dataset}/{category}: inventory COMPLETE | elapsed {time.monotonic() - inventoryStarted:.1f}s', flush=True)
        inventoryComplete[dataset] = True
    except Exception as error:
        addIssue(dataset, '', '', datasetRoots[dataset], f'Inventory incomplete: {type(error).__name__}: {error}')
for dataset in datasetRoots:
    print(f'{dataset}: inventory stage COMPLETE | complete={inventoryComplete[dataset]}', flush=True)
    print(f'{dataset}: {sum(r["imageCount"] for r in inventory if r["dataset"] == dataset)} images inventoried')


# 3 — Dataset Integrity
Check **all** CSV/image/mask references by filename metadata. Decode a fixed-seed random sample of **up to 15 images per category across all groups**, plus **up to 15 available masks per category**. Image sampling rotates through shuffled split/class groups before filling the budget, covering each nonempty group where feasible. Samples are reproducible for the same relative filename inventory and seed.
For sampled masks, check one-to-one correspondence and paired-image dimensions. Paired images outside the image sample have only their headers read for dimensions, **not their pixels decoded**. Missing or withheld mask coverage is a reported limitation; broken explicit CSV references, duplicate annotations, and unmatched available masks are critical errors. VisA masks may contain multiple anomaly IDs. Start/completion messages and flushed progress every five sampled files show elapsed time.


In [ ]:
print('Annotation checks START (all records; metadata only)', flush=True)
annotationStarted = time.monotonic()
allImagePaths = {r[3].absolute() for r in imageRecords}
allMaskPaths = {r[3].absolute() for r in maskRecords}
inventoriedPaths = allImagePaths | allMaskPaths

def resolveReference(value, categoryRoot):
    value = value.strip().replace('\\', '/')
    relativePath = Path(value)
    if not value or relativePath.is_absolute() or '..' in relativePath.parts:
        raise ValueError(f'Invalid relative reference: {value!r}')
    candidates = {(base / relativePath).absolute() for base in [categoryRoot, categoryRoot.parent]
                  if (base / relativePath).absolute() in inventoriedPaths}
    candidates = {p for p in candidates if p.is_relative_to(categoryRoot.absolute())}
    if len(candidates) != 1:
        raise ValueError(f'Reference missing, outside category, or ambiguous: {value}')
    return candidates.pop()

for categoryRoot in visaCategoryRoots:
    category = categoryRoot.name
    print(f'VisA/{category}: annotation checks START', flush=True)
    try:
        csvPaths = [p for p in [categoryRoot / 'image_anno.csv', categoryRoot / 'image_annot.csv'] if p.is_file()]
        if len(csvPaths) != 1:
            raise ValueError('Expected exactly one per-category annotation CSV.')
        categoryImages = {p.absolute(): partition for dataset, cat, partition, p in imageRecords if dataset == 'VisA' and cat == category}
        seenImages = set()
        with csvPaths[0].open(newline='', encoding='utf-8-sig') as handle:
            reader = csv.DictReader(handle)
            headerMap = {str(h).strip().lower(): h for h in (reader.fieldnames or [])}
            imageKey = next((headerMap[k] for k in ['image', 'image_path'] if k in headerMap), None)
            maskKey = next((headerMap[k] for k in ['mask', 'mask_path'] if k in headerMap), None)
            labelKey = headerMap.get('label')
            if not all([imageKey, maskKey, labelKey]):
                raise ValueError(f'Unrecognized annotation columns: {reader.fieldnames}')
            for rowNumber, row in enumerate(reader, 2):
                try:
                    imagePath = resolveReference(row[imageKey], categoryRoot)
                    label = row[labelKey].strip().lower()
                    partition = categoryImages.get(imagePath)
                    # Anomaly labels can name defect subtypes; the image path defines the group.
                    if (partition not in ['Normal', 'Anomaly'] or not label
                            or (partition == 'Normal' and label != 'normal')
                            or (partition == 'Anomaly' and label == 'normal')):
                        raise ValueError('Invalid label or image reference disagrees with image group.')
                    if imagePath in seenImages:
                        raise ValueError('Duplicate image annotation.')
                    seenImages.add(imagePath)
                    annotationCounts[('VisA', category, partition)] += 1
                    maskValue = (row[maskKey] or '').strip()
                    if maskValue.lower() in ['', 'none', 'nan']:
                        maskValue = ''
                    if partition == 'Anomaly' and not maskValue:
                        addIssue('VisA', category, partition, imagePath, 'Mask annotation unavailable; no pixel-level qualification for this image.', 'limitation')
                    if maskValue:
                        maskPath = resolveReference(maskValue, categoryRoot)
                        if not maskPath.is_relative_to((categoryRoot / 'Data' / 'Masks').absolute()):
                            raise ValueError('Mask reference is outside Data/Masks.')
                        maskPairs.append(('VisA', category, partition, imagePath, maskPath))
                except Exception as error:
                    addIssue('VisA', category, '', csvPaths[0], f'CSV row {rowNumber}: {error}')
        for imagePath in categoryImages.keys() - seenImages:
            addIssue('VisA', category, categoryImages[imagePath], imagePath, 'Image has no valid CSV annotation.')
    except Exception as error:
        addIssue('VisA', category, '', categoryRoot, f'Annotation validation failed: {error}')
    print(f'VisA/{category}: annotation checks COMPLETE | elapsed {time.monotonic() - annotationStarted:.1f}s', flush=True)


# Check every available mask's correspondence without opening its contents.
pairedMasks = Counter()
maskImagePaths = {}
for dataset, category, partition, imagePath, maskPath in maskPairs:
    imagePath, maskPath = imagePath.absolute(), maskPath.absolute()
    if maskPath not in allMaskPaths:
        addIssue(dataset, category, partition, maskPath,
                 'Ground-truth mask unavailable; excluded from pixel-level qualification.', 'limitation')
        continue
    pairedMasks[(dataset, maskPath)] += 1
    maskImagePaths[(dataset, maskPath)] = imagePath
for dataset, category, partition, maskPath in maskRecords:
    count = pairedMasks[(dataset, maskPath.absolute())]
    if count != 1:
        addIssue(dataset, category, partition, maskPath, f'Mask must match exactly one image; found {count}.')
print(f'Annotation checks COMPLETE | elapsed {time.monotonic() - annotationStarted:.1f}s', flush=True)

def selectSample(records, dataset, category, recordType):
    # Independent RNG per category/type; no absolute paths or Python hash() in seed.
    rng = random.Random(f'{sampleSeed}|{dataset}|{category}|{recordType}')
    groups = {}
    for record in sorted(set(records), key=lambda r: (r[2], r[3].as_posix())):
        if record[:2] == (dataset, category):
            groups.setdefault(record[2], []).append(record)
    groupNames = sorted(groups)
    rng.shuffle(groupNames)
    for group in groups.values():
        rng.shuffle(group)
    selected = []
    while len(selected) < sampleLimit and any(groups.values()):
        for groupName in groupNames:
            if groups[groupName] and len(selected) < sampleLimit:
                selected.append(groups[groupName].pop())
    return selected

imageSizes = {}
for dataset in datasetRoots:
    categories = sorted({r['category'] for r in inventory if r['dataset'] == dataset})
    for category in categories:
        for recordType, records, selectedRecords in [('images', imageRecords, sampledImageRecords),
                                                      ('masks', maskRecords, sampledMaskRecords)]:
            selected = selectSample(records, dataset, category, recordType)
            selectedRecords.extend(selected)
            if len(selected) < sampleLimit:
                addIssue(dataset, category, '', accessibleRoots.get(dataset, datasetRoots[dataset]) / category,
                         f'Only {len(selected)} {recordType} available for the {sampleLimit}-file sample; all available selected.', 'limitation')
            stageStarted = time.monotonic()
            print(f'{dataset}/{category}: sampled {recordType} START | 0/{len(selected)} | elapsed 0.0s', flush=True)
            for processed, (_, _, partition, filePath) in enumerate(selected, 1):
                filesChecked[dataset][recordType] += 1  # Attempts, including failures.
                try:
                    with Image.open(filePath) as image:
                        image.verify()
                    with Image.open(filePath) as image:
                        image.load()
                        dimensions = image.size
                    if recordType == 'images':
                        imageSizes[filePath.absolute()] = dimensions
                    else:
                        pairedImage = maskImagePaths.get((dataset, filePath.absolute()))
                        if pairedImage is not None:
                            if pairedImage not in imageSizes:
                                # Header access only; preserve the 15-image pixel-decode cap.
                                filesChecked[dataset]['pairedImageHeaders'] += 1
                                with Image.open(pairedImage) as image:
                                    imageSizes[pairedImage] = image.size
                            if imageSizes[pairedImage] != dimensions:
                                addIssue(dataset, category, partition, filePath,
                                         f'Image/mask dimensions disagree: {imageSizes[pairedImage]} versus {dimensions}.')
                except Exception as error:
                    addIssue(dataset, category, partition, filePath,
                             f'Sampled {recordType} integrity check failed: {type(error).__name__}: {error}')
                if processed % 5 == 0:
                    print(f'{dataset}/{category}: sampled {recordType} | {processed}/{len(selected)} | elapsed {time.monotonic() - stageStarted:.1f}s', flush=True)
            print(f'{dataset}/{category}: sampled {recordType} COMPLETE | {len(selected)}/{len(selected)} | elapsed {time.monotonic() - stageStarted:.1f}s', flush=True)
    integrityComplete[dataset] = inventoryComplete[dataset]
samplesComplete = True

# Preserve complete filename evidence, grouped by dataset/category/partition.
inventoryIndex = {(r['dataset'], r['category'], r['partition']): dict(r) for r in inventory}
for record in imageRecords + maskRecords:
    key = record[:3]
    inventoryIndex.setdefault(key, dict(dataset=key[0], category=key[1], partition=key[2]))
for key, row in inventoryIndex.items():
    dataset, category, partition = key
    for column, records in [('imagePaths', imageRecords), ('maskPaths', maskRecords),
                            ('sampledImagePaths', sampledImageRecords), ('sampledMaskPaths', sampledMaskRecords)]:
        paths = sorted({str(r[3].relative_to(accessibleRoots[dataset])) for r in records if r[:3] == key})
        row[column] = json.dumps(paths)
        countColumn = {'imagePaths': 'imageCount', 'maskPaths': 'maskCount',
                       'sampledImagePaths': 'sampledImageCount', 'sampledMaskPaths': 'sampledMaskCount'}[column]
        row[countColumn] = len(paths)
    row['annotationCount'] = annotationCounts[key]
    if dataset == 'MVTec AD 2' and partition in ['train/good', 'validation/good', 'test_public/good', 'test_public/bad']:
        row['annotationCount'] = row['imageCount']  # Directory-encoded labels.
inventory = list(inventoryIndex.values())


# 4 — Acceptance and Evidence
Every PASS means **sample-based qualification**, never exhaustive image integrity certification. All required image directories, filenames, and annotation references are checked; only selected image/mask contents are decoded. **Unsampled corrupt or inconsistent files can remain undetected.** Missing mask coverage and private image exclusions limit the qualified scope and are recorded separately from critical errors. Installed versions are not inferred from folder names.
The three artifacts include full filename/count evidence, sample paths/counts, seed, errors, limitations, and completion status. Interrupted execution leaves incomplete FAIL evidence. Carlos runs Colab; Sam Chatley reviews the evidence; Carlos retains final acceptance authority.


In [ ]:
print('Evidence writing START', flush=True)
validationSummary = {
    'qualificationBasis': qualificationBasis,
    'residualRisk': residualRisk,
    'sampleSeed': sampleSeed,
    'sampleLimitPerCategory': sampleLimit,
    'samplingMethod': 'Seeded shuffled groups, round-robin without replacement; independent category/type RNGs.',
    'validationTimestampUtc': datetime.now(timezone.utc).isoformat(),
    'runStartedUtc': runStartedUtc,
    'complete': all(inventoryComplete.values()) and all(integrityComplete.values()) and samplesComplete,
    'runtime': {'python': platform.python_version(), 'pillow': PIL.__version__},
    'datasets': {}}
for dataset, configuredRoot in datasetRoots.items():
    datasetRows = [r for r in inventory if r['dataset'] == dataset]
    criticalCount = sum(e['dataset'] == dataset and e['severity'] == 'critical' for e in exceptions)
    complete = inventoryComplete[dataset] and integrityComplete[dataset] and samplesComplete
    datasetLimitations = limitations[dataset] + [e['issueDescription'] + ' ' + e['filePath'] for e in exceptions if e['dataset'] == dataset and e['severity'] == 'limitation']
    validationSummary['datasets'][dataset] = {
        'configuredRoot': str(configuredRoot),
        'accessibleRoot': str(accessibleRoots[dataset]) if dataset in accessibleRoots else None,
        'releaseIdentifier': None,
        'releaseNote': ('MVTec AD 2; installed release not independently verified.' if dataset == 'MVTec AD 2' else 'Official AWS archive is VisA_20220922.tar; installed release not independently verified.'),
        'complete': complete,
        'qualificationBasis': qualificationBasis,
        'fullInventoryTotals': {'images': sum(r['imageCount'] for r in datasetRows), 'masks': sum(r['maskCount'] for r in datasetRows), 'annotations': sum(r['annotationCount'] for r in datasetRows)},
        'sampleSizes': {'images': sum(r['sampledImageCount'] for r in datasetRows), 'masks': sum(r['sampledMaskCount'] for r in datasetRows)},
        'categorySampleCounts': {c: {'images': sum(r['sampledImageCount'] for r in datasetRows if r['category'] == c), 'masks': sum(r['sampledMaskCount'] for r in datasetRows if r['category'] == c)} for c in sorted({r['category'] for r in datasetRows})},
        'result': 'PASS' if complete and criticalCount == 0 else 'FAIL',
        'filesChecked': dict(filesChecked[dataset]),
        'categoryCount': len({r['category'] for r in datasetRows if r['imageCount']}),
        'partitionCount': len({r['partition'] for r in datasetRows if r['imageCount']}),
        'categoryPartitionCount': sum(r['imageCount'] > 0 for r in datasetRows),
        'criticalFailureCount': criticalCount,
        'annotationAvailabilityOrLimitations': datasetLimitations,
        'primaryResearchCategories': ['can', 'wallplugs', 'sheet_metal'] if dataset == 'MVTec AD 2' else []}
validationSummary['overallResult'] = 'PASS' if all(d['result'] == 'PASS' for d in validationSummary['datasets'].values()) else 'FAIL'
for csvName, rows, fieldNames in [
    ('dataset_inventory.csv', inventory, inventoryFields),
    ('validation_exceptions.csv', exceptions, ['dataset', 'category', 'partition', 'filePath', 'severity', 'issueDescription'])]:
    with (outputRoot / csvName).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldNames)
        writer.writeheader()
        writer.writerows(rows)
# Replace checkpoint only after both CSVs have been saved.
summaryTemp = outputRoot / 'validation_summary.json.tmp'
with summaryTemp.open('w', encoding='utf-8') as handle:
    json.dump(validationSummary, handle, indent=2)
summaryTemp.replace(outputRoot / 'validation_summary.json')
for dataset, result in validationSummary['datasets'].items():
    print(f"{dataset}: {result['result']} — sample-based qualification | {result['sampleSizes']['images']} sampled images / {result['fullInventoryTotals']['images']} inventoried | {result['sampleSizes']['masks']} sampled masks | {result['criticalFailureCount']} critical failures", flush=True)
print('Overall:', validationSummary['overallResult'], '— sample-based qualification', flush=True)
print('Residual risk:', residualRisk, flush=True)
print('Evidence:', outputRoot)

print('Evidence writing COMPLETE', flush=True)
